In [9]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import os
df = pd.read_csv(r"F:\FinalYr Project\nutrisense\data\processed\tamilnadu_nfhs5_merged.csv")
print("Original shape:", df.shape)
print(df.dtypes)

Original shape: (6498, 13)
caseid               object
midx                  int64
v001                  int64
v002                  int64
state                 int64
stunting_haz        float64
mother_education      int64
wealth_index          int64
mother_bmi          float64
hhid                  int64
toilet_type           int64
electricity           int64
share_toilet        float64
dtype: object


In [10]:
# ── Cell 2: Define target ───────────────────────────────────────────────
# Target: binary stunting label (1 = stunted, 0 = not stunted)
df['stunted'] = (df['stunting_haz'] < -2.0).astype(int)
print("Stunting prevalence:", df['stunted'].mean().round(3))

Stunting prevalence: 0.721


In [11]:
# ── Cell 3: Drop ID columns (not useful for prediction) ─────────────────
# caseid, midx, v001, v002, hhid are identifiers — they don't carry
# predictive signal and would cause data leakage if included
drop_cols = ['caseid', 'midx', 'v001', 'v002', 'hhid', 'stunting_haz']
df_model = df.drop(columns=drop_cols)
print("Columns after drop:", df_model.columns.tolist())

Columns after drop: ['state', 'mother_education', 'wealth_index', 'mother_bmi', 'toilet_type', 'electricity', 'share_toilet', 'stunted']


In [12]:
# ── Cell 4: Handle missing values ───────────────────────────────────────
# Strategy: fill numeric with median, categorical with mode
# (simple but robust for a first model)
for col in df_model.columns:
    if df_model[col].dtype == 'object':
        df_model[col].fillna(df_model[col].mode()[0], inplace=True)
    else:
        df_model[col].fillna(df_model[col].median(), inplace=True)

print("Any nulls remaining:", df_model.isnull().sum().sum())

Any nulls remaining: 0


C:\Users\dhanu\AppData\Local\Temp\ipykernel_23004\4072237436.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_model[col].fillna(df_model[col].median(), inplace=True)
C:\Users\dhanu\AppData\Local\Temp\ipykernel_23004\4072237436.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy

In [13]:
# ── Cell 5: Encode categorical columns ──────────────────────────────────
# LabelEncoder turns text categories into integer codes
# e.g., 'primary' -> 1, 'secondary' -> 2, 'higher' -> 3
le = LabelEncoder()
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:", cat_cols)

for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print("Dtypes after encoding:")
print(df_model.dtypes)

Categorical columns to encode: []
Dtypes after encoding:
state                 int64
mother_education      int64
wealth_index          int64
mother_bmi          float64
toilet_type           int64
electricity           int64
share_toilet        float64
stunted               int32
dtype: object


In [14]:
# ── Cell 6: Split features and target ───────────────────────────────────
X = df_model.drop(columns=['stunted'])
y = df_model['stunted']

print("Feature matrix X shape:", X.shape)
print("Target y distribution:\n", y.value_counts())

Feature matrix X shape: (6498, 7)
Target y distribution:
 stunted
1    4684
0    1814
Name: count, dtype: int64


In [16]:
# ── Cell 7: Save model-ready data ───────────────────────────────────────
import os

# Define output path using the absolute project path
output_path = r"F:\FinalYr Project\nutrisense\data\processed\tamilnadu_model_ready.csv"

# Create directory if it doesn't exist and save
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_model.to_csv(output_path, index=False)
print("Saved model-ready dataset!")


Saved model-ready dataset!
